In [1]:
import pandas as pd
import altair as alt

##### **Reading in the data (Goodreads Top Ranked Novels and *New York Times* Bestseller datasets)**

In [2]:
novels_df = pd.read_csv("https://raw.githubusercontent.com/melaniewalsh/responsible-datasets-in-context/main/datasets/top-500-novels/final_merged_dataset_no_full_text.tsv", sep="\t")
novels_df.head()

,top_500_rank,title,author,pub_year,orig_lang,genre,author_birth,author_death,author_gender,author_primary_lang,...,gr_num_ratings,gr_num_reviews,gr_avg_rating_rank,gr_num_ratings_rank,oclc_owi,author_viaf,gr_url,wiki_url,pg_eng_url,pg_orig_url
0,1,Don Quixote,Miguel de Cervantes,1605,Spanish,action,1547,1616,male,spa,...,"269,435","12,053",318,211,1.810748e+09,17220427,https://www.goodreads.com/book/show/3836.Don_Q...,https://en.wikipedia.org/wiki/Don_Quixote,https://www.gutenberg.org/cache/epub/996/pg996...,https://www.gutenberg.org/cache/epub/2000/pg20...
1,2,Alice's Adventures in Wonderland,Lewis Carroll,1865,English,fantasy,1832,1898,male,eng,...,"561,016","15,380",172,133,1.156132e+10,66462036,https://www.goodreads.com/book/show/24213.Alic...,https://en.wikipedia.org/wiki/Alice%27s_Advent...,https://www.gutenberg.org/cache/epub/11/pg11.txt,NaN
2,3,The Adventures of Huckleberry Finn,Mark Twain,1884,English,action,1835,1910,male,eng,...,"1,262,480","19,440",373,68,3.373178e+09,50566653,https://www.goodreads.com/book/show/2956.The_A...,https://en.wikipedia.org/wiki/Adventures_of_Hu...,https://www.gutenberg.org/cache/epub/76/pg76.txt,NaN
3,4,The Adventures of Tom Sawyer,Mark Twain,1876,English,action,1835,1910,male,eng,...,"931,898","13,603",301,88,3.373178e+09,50566653,https://www.goodreads.com/book/show/24583.The_...,https://en.wikipedia.org/wiki/The_Adventures_o...,https://www.gutenberg.org/cache/epub/74/pg74.txt,NaN
4,5,Treasure Island,Robert Louis Stevenson,1883,English,action,1850,1894,male,eng,...,"486,155","16,307",368,145,3.434000e+03,95207986,https://www.goodreads.com/book/show/295.Treasu...,https://en.wikipedia.org/wiki/Treasure_Island,https://www.gutenberg.org/cache/epub/120/pg120...,NaN


In [3]:
nyt_bestsellers_df = pd.read_csv("https://raw.githubusercontent.com/ecds/post45-datasets/main/nyt_full.tsv", sep="\t", encoding='utf-8')
nyt_bestsellers_df.head()

,year,week,rank,title_id,title,author
0,1931,1931-10-12,1,6477,THE TEN COMMANDMENTS,Warwick Deeping
1,1931,1931-10-12,2,1808,FINCHE'S FORTUNE,Mazo de la Roche
2,1931,1931-10-12,3,5304,THE GOOD EARTH,Pearl S. Buck
3,1931,1931-10-12,4,4038,SHADOWS ON THE ROCK,Willa Cather
4,1931,1931-10-12,5,3946,SCARMOUCHE THE KING MAKER,Rafael Sabatini


In [4]:
novels_df.shape

(500, 29)

In [5]:
nyt_bestsellers_df.shape

(60386, 6)

##### **Checking for Missing Data**

In [6]:
novels_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   top_500_rank              500 non-null    int64  
 1   title                     500 non-null    str    
 2   author                    500 non-null    str    
 3   pub_year                  500 non-null    int64  
 4   orig_lang                 500 non-null    str    
 5   genre                     500 non-null    str    
 6   author_birth              499 non-null    str    
 7   author_death              496 non-null    str    
 8   author_gender             500 non-null    str    
 9   author_primary_lang       500 non-null    str    
 10  author_nationality        500 non-null    str    
 11  author_field_of_activity  329 non-null    str    
 12  author_occupation         459 non-null    str    
 13  oclc_holdings             495 non-null    float64
 14  oclc_eholdings       

In [7]:
nyt_bestsellers_df.isna().sum()

year         0
week         0
rank         0
title_id     0
title        0
author      10
dtype: int64

In [8]:
novels_df.isna().sum()

top_500_rank                  0
title                         0
author                        0
pub_year                      0
orig_lang                     0
genre                         0
author_birth                  1
author_death                  4
author_gender                 0
author_primary_lang           0
author_nationality            0
author_field_of_activity    171
author_occupation            41
oclc_holdings                 5
oclc_eholdings                5
oclc_total_editions           5
oclc_holdings_rank            5
oclc_editions_rank            5
gr_avg_rating                 0
gr_num_ratings                0
gr_num_reviews                0
gr_avg_rating_rank            0
gr_num_ratings_rank           0
oclc_owi                      5
author_viaf                   0
gr_url                       20
wiki_url                      0
pg_eng_url                    0
pg_orig_url                 436
dtype: int64

In [9]:
# Calculate percentage for all columns for novels
missing_pct_novels = (novels_df.isna().sum() / len(novels_df)) * 100

# Filter to show only columns that actually have missing data
print(missing_pct_novels[missing_pct_novels > 0].sort_values(ascending=False))

pg_orig_url                 87.2
author_field_of_activity    34.2
author_occupation            8.2
gr_url                       4.0
oclc_holdings_rank           1.0
oclc_total_editions          1.0
oclc_holdings                1.0
oclc_eholdings               1.0
oclc_owi                     1.0
oclc_editions_rank           1.0
author_death                 0.8
author_birth                 0.2
dtype: float64


- less than or equal to 1% of missing data: rows will be dropped
- greater than or equal to 5% but less than or equal to 70% of missing data: impute missing data
- greater than 70% of missing data: column will be dropped

In [10]:
rows_drop = missing_pct_novels[(missing_pct_novels <= 1) & (missing_pct_novels > 0)]
rows_drop

author_birth           0.2
author_death           0.8
oclc_holdings          1.0
oclc_eholdings         1.0
oclc_total_editions    1.0
oclc_holdings_rank     1.0
oclc_editions_rank     1.0
oclc_owi               1.0
dtype: float64

In [11]:
columns_rows_drop = ['author_birth', 'author_death', 'oclc_holdings', 'oclc_eholdings',
                     'oclc_total_editions', 'oclc_holdings_rank', 'oclc_editions_rank', 'oclc_owi']

In [12]:
novels_df = novels_df.dropna(subset=columns_rows_drop)

In [13]:
novels_df.shape

(491, 29)

In [14]:
novels_df = novels_df.drop(columns=['pg_orig_url'])

In [15]:
novels_df.shape

(491, 28)

In [16]:
columns_to_impute =  missing_pct_novels[(missing_pct_novels >= 5) & (missing_pct_novels <= 70)]
columns_to_impute

author_field_of_activity    34.2
author_occupation            8.2
dtype: float64

This may not be the most appropriate imputation method, but I've decided to impute by the mode just for this assignment.

In [17]:
novels_df['author_field_of_activity'].head()

0                                         NaN
1    writing,teaching,photography,mathematics
2              wit and humor,literature,humor
3              wit and humor,literature,humor
4                                         NaN
Name: author_field_of_activity, dtype: str

In [18]:
novels_df['author_occupation'].head()

0    soldiers,poets spanish,novelists spanish,drama...
1        authors,teachers,photographers,mathematicians
2                          lecturers,humorists,authors
3                          lecturers,humorists,authors
4                       travel writers,poets,novelists
Name: author_occupation, dtype: str

In [19]:
mode_val = novels_df['author_field_of_activity'].mode()[0]
mode_val

'english literature'

In [20]:
novels_df = novels_df.fillna(novels_df['author_field_of_activity'].mode()[0])
novels_df.head()

,top_500_rank,title,author,pub_year,orig_lang,genre,author_birth,author_death,author_gender,author_primary_lang,...,gr_avg_rating,gr_num_ratings,gr_num_reviews,gr_avg_rating_rank,gr_num_ratings_rank,oclc_owi,author_viaf,gr_url,wiki_url,pg_eng_url
0,1,Don Quixote,Miguel de Cervantes,1605,Spanish,action,1547,1616,male,spa,...,3.90,"269,435","12,053",318,211,1.810748e+09,17220427,https://www.goodreads.com/book/show/3836.Don_Q...,https://en.wikipedia.org/wiki/Don_Quixote,https://www.gutenberg.org/cache/epub/996/pg996...
1,2,Alice's Adventures in Wonderland,Lewis Carroll,1865,English,fantasy,1832,1898,male,eng,...,4.06,"561,016","15,380",172,133,1.156132e+10,66462036,https://www.goodreads.com/book/show/24213.Alic...,https://en.wikipedia.org/wiki/Alice%27s_Advent...,https://www.gutenberg.org/cache/epub/11/pg11.txt
2,3,The Adventures of Huckleberry Finn,Mark Twain,1884,English,action,1835,1910,male,eng,...,3.83,"1,262,480","19,440",373,68,3.373178e+09,50566653,https://www.goodreads.com/book/show/2956.The_A...,https://en.wikipedia.org/wiki/Adventures_of_Hu...,https://www.gutenberg.org/cache/epub/76/pg76.txt
3,4,The Adventures of Tom Sawyer,Mark Twain,1876,English,action,1835,1910,male,eng,...,3.92,"931,898","13,603",301,88,3.373178e+09,50566653,https://www.goodreads.com/book/show/24583.The_...,https://en.wikipedia.org/wiki/The_Adventures_o...,https://www.gutenberg.org/cache/epub/74/pg74.txt
4,5,Treasure Island,Robert Louis Stevenson,1883,English,action,1850,1894,male,eng,...,3.84,"486,155","16,307",368,145,3.434000e+03,95207986,https://www.goodreads.com/book/show/295.Treasu...,https://en.wikipedia.org/wiki/Treasure_Island,https://www.gutenberg.org/cache/epub/120/pg120...


In [21]:
novels_cleaned = novels_df.fillna(novels_df['author_occupation'].mode()[0])
novels_cleaned.isna().sum().sum()

np.int64(0)

In [22]:
# Calculate percentage for all columns for nyt
missing_pct_nyt = (nyt_bestsellers_df.isna().sum() / len(nyt_bestsellers_df)) * 100

# Filter to show only columns that actually have missing data
print(missing_pct_nyt[missing_pct_nyt > 0].sort_values(ascending=False))

author    0.01656
dtype: float64


In [23]:
nyt_bestsellers_cleaned = nyt_bestsellers_df.dropna(subset=['author'])

In [24]:
nyt_bestsellers_cleaned.isna().sum().sum()

np.int64(0)

##### **Merging the Datasets**

In [25]:
set(novels_cleaned.columns).intersection(set(nyt_bestsellers_cleaned.columns))

{'author', 'title'}

In [26]:
novels_cleaned.shape

(491, 28)

In [27]:
nyt_bestsellers_cleaned.shape

(60376, 6)

Instead of conducting a left join merge as we have done in class, I want to perform an inner join merge since I only want the entries from both of my datasets to match to one another instead of only matching to the left dataset (novels_cleaned). If I perform a left-join merge, I know that I'll have missing data in the right-side of the merged dataset from nyt_bestsellers_cleaned. I don't want any missing information when I perform EDA, so I will take the cautionary to save the trouble by doing an inner-join merge. 

In [28]:
merged_df = novels_cleaned.merge(nyt_bestsellers_cleaned, how='inner', on=['author', 'title'])

In [29]:
merged_df.shape

(0, 32)

In [30]:
shared_authors = novels_cleaned[novels_cleaned['author'].isin(nyt_bestsellers_cleaned['author'])]['author'].nunique()
shared_titles = novels_cleaned[novels_cleaned['title'].isin(nyt_bestsellers_cleaned['title'])]['title'].nunique()

In [31]:
print(f"Number of shared authors: {shared_authors}")
print(f"Number of shared titles: {shared_titles}")

Number of shared authors: 119
Number of shared titles: 0


As we have done in class, let's look further into the titles to see why there are no shared titles between the two datasets. 

In [32]:
novels_cleaned[novels_cleaned['author'].isin(nyt_bestsellers_cleaned['author'])]['author'].value_counts().head()

author
John Grisham       19
John Steinbeck      8
Nicholas Sparks     7
Stephen King        7
J.R.R. Tolkien      5
Name: count, dtype: int64

In [33]:
nyt_bestsellers_cleaned[nyt_bestsellers_cleaned['author'].isin(novels_cleaned['author'])]['author'].value_counts().head()

author
Stephen King       892
John Grisham       789
David Baldacci     396
Nicholas Sparks    390
Herman Wouk        375
Name: count, dtype: int64

In [34]:
novel_titles = novels_cleaned[novels_cleaned.author == "Stephen King"].title.unique()
nyt_titles = nyt_bestsellers_cleaned[nyt_bestsellers_cleaned.author == "Stephen King"].title.unique()

print(f"Stephen King's novels: {novel_titles}")
print(f"Stephen King's NYT bestsellers: {nyt_titles}")

Stephen King's novels: <ArrowStringArray>
[     'The Stand',             'It',    'The Shining',         'Misery',
         'Carrie', 'The Gunslinger',   'Dreamcatcher']
Length: 7, dtype: str
Stephen King's NYT bestsellers: <ArrowStringArray>
[                  'THE SHINING',                     'THE STAND',
                 'THE DEAD ZONE',                   'FIRESTARTER',
                          'CUJO',             'DIFFERENT SEASONS',
                     'CHRISTINE',                  'PET SEMATARY',
                 'SKELETON CREW',             'THE BACHMAN BOOKS',
                            'IT',        'THE EYES OF THE DRAGON',
                        'MISERY',             'THE TOMMYKNOCKERS',
                 'THE DARK HALF',            'FOUR PAST MIDNIGHT',
                'NEEDFUL THINGS',                 'GERALD'S GAME',
             'DOLORES CLAIBORNE',      'NIGHTMARES & DREAMSCAPES',
                      'INSOMNIA',                   'ROSE MADDER',
                   '

In [35]:
nyt_bestsellers_cleaned = nyt_bestsellers_cleaned.rename(columns={'title': 'nyt_title'})
nyt_bestsellers_cleaned['title'] = nyt_bestsellers_cleaned['nyt_title'].str.capitalize()

In [36]:
shared_titles = novels_cleaned[novels_cleaned['title'].isin(nyt_bestsellers_cleaned['title'])]['title'].nunique()
print(f"Number of shared titles: {shared_titles}")

Number of shared titles: 12


In [37]:
merged_df = novels_cleaned.merge(nyt_bestsellers_cleaned, how='inner', on=['author', 'title'])

In [38]:
merged_df.shape

(231, 33)

Though we can explore further using some string methods learned in class, let's get to EDA with the combined dataset we have now to identify any patterns, gaps, and potential biases that can inform future research on these novels.

##### **EDA**

In [39]:
merged_df.shape

(231, 33)

In [40]:
merged_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 231 entries, 0 to 230
Data columns (total 33 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   top_500_rank              231 non-null    int64  
 1   title                     231 non-null    str    
 2   author                    231 non-null    str    
 3   pub_year                  231 non-null    int64  
 4   orig_lang                 231 non-null    str    
 5   genre                     231 non-null    str    
 6   author_birth              231 non-null    str    
 7   author_death              231 non-null    str    
 8   author_gender             231 non-null    str    
 9   author_primary_lang       231 non-null    str    
 10  author_nationality        231 non-null    str    
 11  author_field_of_activity  231 non-null    str    
 12  author_occupation         231 non-null    str    
 13  oclc_holdings             231 non-null    float64
 14  oclc_eholdings       

In [41]:
merged_df.head()

,top_500_rank,title,author,pub_year,orig_lang,genre,author_birth,author_death,author_gender,author_primary_lang,...,oclc_owi,author_viaf,gr_url,wiki_url,pg_eng_url,year,week,rank,title_id,nyt_title
0,47,Ulysses,James Joyce,1922,English,na,1882,1941,male,eng,...,395796.0,44300643,https://www.goodreads.com/book/show/338798.Uly...,https://en.wikipedia.org/wiki/Ulysses_(novel),https://www.gutenberg.org/cache/epub/4300/pg43...,1934,1934-02-05,6,7041,ULYSSES
1,47,Ulysses,James Joyce,1922,English,na,1882,1941,male,eng,...,395796.0,44300643,https://www.goodreads.com/book/show/338798.Uly...,https://en.wikipedia.org/wiki/Ulysses_(novel),https://www.gutenberg.org/cache/epub/4300/pg43...,1934,1934-02-12,2,7041,ULYSSES
2,47,Ulysses,James Joyce,1922,English,na,1882,1941,male,eng,...,395796.0,44300643,https://www.goodreads.com/book/show/338798.Uly...,https://en.wikipedia.org/wiki/Ulysses_(novel),https://www.gutenberg.org/cache/epub/4300/pg43...,1934,1934-02-19,6,7041,ULYSSES
3,47,Ulysses,James Joyce,1922,English,na,1882,1941,male,eng,...,395796.0,44300643,https://www.goodreads.com/book/show/338798.Uly...,https://en.wikipedia.org/wiki/Ulysses_(novel),https://www.gutenberg.org/cache/epub/4300/pg43...,1934,1934-02-26,3,7041,ULYSSES
4,47,Ulysses,James Joyce,1922,English,na,1882,1941,male,eng,...,395796.0,44300643,https://www.goodreads.com/book/show/338798.Uly...,https://en.wikipedia.org/wiki/Ulysses_(novel),https://www.gutenberg.org/cache/epub/4300/pg43...,1934,1934-03-05,4,7041,ULYSSES


In [42]:
merged_df['genre'].value_counts()

genre
horror       80
na           59
romance      32
history      31
thrillers    29
Name: count, dtype: int64

In [43]:
merged_df['gr_num_ratings_rank'].value_counts()

gr_num_ratings_rank
78     35
125    32
158    31
116    30
135    29
405    18
101    18
258    15
141    14
289     9
Name: count, dtype: int64

In [44]:
alt.Chart(merged_df).mark_bar().encode(
    x=alt.X("genre:N", sort="-y", title='Genre'),
    y=alt.Y("mean(gr_num_ratings_rank):Q", title='Avg Goodreads Ranking')
).properties(
    width=600,
    height=400,
    title="Average Popularity Rank by Genre"
)

alt.Chart(...)

Out of the 231 novels, the majority of the novels are categorized as "na" genre. From my understanding, "na" in this context means the novel does not have a genre listed. For further and reliable analysis of how the genre of the novel may affect its Goodreads ranking, it would be benefitical to get more data on those novels. 

In [45]:
alt.Chart(merged_df).mark_bar().encode(
    x=alt.X("author_gender:N", title="Author Gender"),
    y=alt.Y('count():Q', stack='normalize', title='Percentage of Novels'),
    color=alt.Color('genre:N', title="Genre"),
    tooltip=['author_gender', 'genre', 'count()']
).properties(
    width=400,
    height=300,
    title="Genre Distribution by Author Gender"
)

alt.Chart(...)

I wanted to see if based on the gender of the author, there is a higher distribution of a specific novel genre. Here we can see the only common genre shared between the two genders is "na." The difference I can note from this visualization is that the romance and history novels are majorly written by female authors while horror and thrillers novels are majorly written by male authors. This tells me the current dataset may reflect traditional publishing silos or sampling bias, rather than the diverse representation of the literary landscape. I would like to note that based on how I merged the two datasets, I could've excluded the representative data I am alluding to. 

In [46]:
alt.Chart(merged_df).mark_bar().encode(
    x=alt.X("author_nationality:N", title="Author Nationality"),
    y=alt.Y('count():Q', stack='normalize', title='Percentage of Books'),
    color=alt.Color('genre:N', title="Genre"),
    tooltip=['author_nationality', 'genre', 'count()']
).properties(
    width=400,
    height=300,
    title="Genre Distribution by Author Nationality"
)

alt.Chart(...)

In [47]:
countries = merged_df['author_nationality'].value_counts()
countries

author_nationality
US    158
GB     46
IE     27
Name: count, dtype: int64

In [48]:
proportions = merged_df['author_nationality'].value_counts(normalize=True)
proportions

author_nationality
US    0.683983
GB    0.199134
IE    0.116883
Name: proportion, dtype: float64

The visualization above shows that not only within the subset of the 500 top novels include only three different nationalities of the authors, but American authors seem to be the most representative of the different genres their novels cover (i.e. 4/5 of the provided genres within the dataset). Upon a further look into the distribution of the nationalities themselves, we can see that the majority of authors are Americans while there is a small and close proportion of British and Irish authors. 

In [49]:
merged_df['pub_date'] = pd.to_datetime(
    merged_df['pub_year'].astype(str) + '-01-01',
    errors='coerce'
)

In [50]:
selection = alt.selection_point(fields=['genre'], bind='legend')

alt.Chart(merged_df).mark_bar().encode(
    x=alt.X('pub_date:T', title='Publication Year'),
    y=alt.Y('count():Q', title='Number of Novels'),
    color=alt.Color('genre:N', title='Genre'),
    tooltip=['genre:N', 'count():Q'],
    opacity=alt.condition(selection, alt.value(1), alt.value(0.2))
).add_params(selection).transform_filter(
    alt.datum.genre != None
).properties(
    title='Top 231 Novels by Genre Over Time',
    width=800,
    height=400
)

alt.Chart(...)

In [51]:
merged_df['pub_year'].value_counts()

pub_year
1987    61
1986    35
1938    32
2001    29
2013    29
2003    18
2010    18
1922     9
Name: count, dtype: int64

The bar chart showing the number of novels across the publication years based on the genres was taken from the class example as I wanted to take a closer look at the visual and give my input based on the subset of data I am working with. From the visual above, we can see that the majority of novels were published around 1986-1987. Also thinking in terms of normality of the data, I can see that the data seems left-skewed, with a good proportion of the data being from "earlier" years than later years. Another observation I have made relates to the genre element included in this visual. Here, we can see there is a noticeable lack of variety within specific years. Instead of seeing a multitude of genres being released, we often see a single common genre dominating a year. Given the small size of this dataset, it makes sense that we may see these "pockets" of data. For example, in 2013, we see that the novels are exclusively Horror rather than a diverse representation of different kinds of novels being published during that time. But then again with the years with "na" novels, those could be that diversity we don't see here if the actual genres were noted. 

In [52]:
merged_df['author_occupation'].value_counts().head(10)

author_occupation
novelists,television writers,television producers and directors,screenwriters,motion picture producers and directors,authors,actors           80
author                                                                                                                                        32
english teachers,college teachers,novelists,authors,writers,university and college faculty members,literature teachers,english teachers of    31
novelists american,authors american                                                                                                           29
screenwriters,novelists,motion picture producers and directors,lawyers,authors,actors                                                         18
novelists,authors,screenwriters,editors,dramatists                                                                                            18
screenwriters,dramatists,authors                                                                                

In [53]:
author_stats = merged_df.groupby('author')['gr_num_ratings_rank'].mean().reset_index()
author_stats

,author,gr_num_ratings_rank
0,Dan Brown,135.0
1,Daphne Du Maurier,125.0
2,Emma Donoghue,101.0
3,Ian McEwan,141.0
4,James Joyce,289.0
5,John Grisham,405.0
6,Stephen King,126.0
7,Toni Morrison,158.0


In [54]:
author_info = merged_df[['author', 'author_gender', 'genre', 'author_occupation']].drop_duplicates(subset='author')

In [55]:
final_author_df = pd.merge(author_info, author_stats, on='author')

In [56]:
final_author_df

,author,author_gender,genre,author_occupation,gr_num_ratings_rank
0,James Joyce,male,na,novelists,289.0
1,Toni Morrison,female,history,"english teachers,college teachers,novelists,au...",158.0
2,Daphne Du Maurier,female,romance,author,125.0
3,Ian McEwan,male,na,"screenwriters,dramatists,authors",141.0
4,Dan Brown,male,thrillers,"novelists american,authors american",135.0
5,Stephen King,male,horror,"novelists,television writers,television produc...",126.0
6,John Grisham,male,na,"screenwriters,novelists,motion picture produce...",405.0
7,Emma Donoghue,female,na,"novelists,authors,screenwriters,editors,dramat...",101.0


While looking at how the string manipulation and melting operation were used to further analyze the data, I wanted to try doing so with the author's occupation. As we can see above, there is a list of occupations an author can have. I want to look at each specific occuptation title and see if there is any connection with gender and genre as well as further explore how the rating may possibly change. 

In [57]:
target_jobs = ['teacher', 'novelists', 'actor', 'screenwriter', 'producer']

# Standardize the cases just in case
final_author_df['clean_occ'] = final_author_df['author_occupation'].str.lower()

for job in target_jobs:
    final_author_df[f'job_{job}'] = final_author_df['clean_occ'].str.contains(job).astype(int)

In [58]:
job_columns = [f'job_{j}' for j in target_jobs]

melted_occupations = pd.melt(
    final_author_df,
    id_vars=['author','genre', 'author_gender', 'gr_num_ratings_rank'],
    value_vars=job_columns,
    var_name='specific_occupation',
    value_name='is_present'
)

# Only keep the rows where the author actually HAS that occupation
melted_occupations = melted_occupations[melted_occupations['is_present'] == 1]

# Clean up the labels (remove 'job_')
melted_occupations['specific_occupation'] = melted_occupations['specific_occupation'].str.replace('job_', '')

In [59]:
melted_occupations.head()

,author,genre,author_gender,gr_num_ratings_rank,specific_occupation,is_present
1,Toni Morrison,history,female,158.0,teacher,1
8,James Joyce,na,male,289.0,novelists,1
9,Toni Morrison,history,female,158.0,novelists,1
12,Dan Brown,thrillers,male,135.0,novelists,1
13,Stephen King,horror,male,126.0,novelists,1


In [60]:
# From 8 authors to 15 because we are doing based on the specific occupation and some authors may have different occupations
melted_occupations.shape

(15, 6)

In [61]:
melted_occupations['specific_occupation'].value_counts()

specific_occupation
novelists       6
screenwriter    4
actor           2
producer        2
teacher         1
Name: count, dtype: int64

In [62]:
melted_occupations['author'].value_counts()

author
Stephen King     4
John Grisham     4
Toni Morrison    2
Emma Donoghue    2
James Joyce      1
Dan Brown        1
Ian McEwan       1
Name: count, dtype: int64

In [63]:
# Aggregate average rank by occupation
rank_occ = melted_occupations.groupby('specific_occupation')['gr_num_ratings_rank'].mean().reset_index()

alt.Chart(rank_occ).mark_point(size=100, filled=True).encode(
    x=alt.X('gr_num_ratings_rank:Q', scale=alt.Scale(zero=False), title="Avg Popularity Rank"),
    y=alt.Y('specific_occupation:N', sort='x', title="Occupation"),
    color=alt.value("#4c78a8")
) + alt.Chart(rank_occ).mark_rule().encode(
    x='gr_num_ratings_rank:Q',
    y='specific_occupation:N'
).properties(title="What Does Career Show About their Novel Popularity?")

alt.LayerChart(...)

From the data, we see writers whose occupations include being an actor and a producer have a stronger novel popularity than other individuals. Before this visual, I have conducted a further investigation of the data itself as I grouped the rows based on the author while also trying to preserve the Goodreads ranking to which I had to create two separate dataframes and merge them. From what I have found, there were in total of 8 authors with some of them having more than one occupation. Those with more than one occupations had resulted in more rows in the melted_occupations dataframe which was made after the melting operation. This means for the rows that represent different occupations, those authors' average Goodreads ranking were repeated. So, the visual may not be entirely accurate with the small sample size we have and the algorithmic approach taken. But also the results could be affected based on the writer's popularity themselves as they expanded their occupation beyond just being a "writer." They may have received success from being a producer, actor, etc. or having their novels adapted into movies.

In [64]:
alt.Chart(melted_occupations).mark_bar().encode(
    # 1. X-axis uses the cleaned occupation labels
    x=alt.X("specific_occupation:N", title="Author Career Background", sort='-y'),
    
    # 2. Normalized Y-axis shows the % breakdown of genres within that job
    y=alt.Y('count():Q', stack='normalize', title='Percentage of Genre Output', axis=alt.Axis(format='%')),
    
    # 3. Color by Genre to see the "Flavor" of each career
    color=alt.Color('genre:N', title="Genre", scale=alt.Scale(scheme='tableau20')),
    
    tooltip=['specific_occupation', 'genre', 'count()']
).properties(
    width=600,
    height=400,
    title="Does Your Day Job Dictate Your Genre?"
)

alt.Chart(...)

Again with the small sample size, the visual may not be entirely representative of the population of the top 500 novels, but from looking at what is shown, we can see that those whose occupation is "novelists" are made up individuals who wrote novels of different genres (i.e. more representative of different kinds of novels) while other occupations tend to be more exclusive to specific novels. With a larger sample size, we may be able to get a better sense how spread out different novel genres are and if there are any patterns. 

In [65]:
alt.Chart(melted_occupations).mark_rect().encode(
    x=alt.X('genre:N', title="Genre"),
    y=alt.Y('specific_occupation:N', title="Occupation"),
    color=alt.Color('count():Q', scale=alt.Scale(scheme='greens'), title="Total Individuals"),
    tooltip=['specific_occupation', 'genre', 'count()']
).properties(title="Genre Specialization by Professional Background")

alt.Chart(...)

From the visualization above, we can get a sense of the number of individuals who resonated with the specific occupation as well as how it maps out to a corresponding genre. 

In [68]:
alt.Chart(melted_occupations).mark_bar().encode(
    x=alt.X('mean(gr_num_ratings_rank):Q', title="Avg Rank"),
    y=alt.Y('author_gender:N', title=None),
    color=alt.Color('author_gender:N', legend=alt.Legend(title='Gender')),
    row=alt.Row('specific_occupation:N', title="Background")
).properties(width=400, height=80, title="Gender Popularity Gap Across Different Careers")

alt.Chart(...)

This representation tells me based on the data provided that those who were identified as actors and producers had the highest average ranking than other individuals of different genders and occupations. There could be an association based on the different heights of the bars, but would be more significant with the inclusion of more data. We can also see that there seems to be more males than females from this specific set of data. 